# 03 - Ingeniería de características temporales

En este cuaderno se generan características temporales a partir de la serie de consumo eléctrico para utilizarlas posteriormente en los modelos de pronóstico.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set(style="whitegrid")

## Carga del dataset y creación de índice temporal

In [ ]:
csv_path = Path("..") / "data" / "household_power_consumption.csv"
df = pd.read_csv(csv_path)

date_col = [c for c in df.columns if c.lower() == "date"][0]
time_col = [c for c in df.columns if c.lower() == "time"][0]

df["datetime"] = pd.to_datetime(df[date_col] + " " + df[time_col], dayfirst=True, errors="coerce")
df = df.set_index("datetime").sort_index()

df.head()

## Resampling a nivel horario y selección de variable objetivo

In [ ]:
gap_col = [c for c in df.columns if c.lower() == "global_active_power"][0]

hourly = df[[gap_col]].resample("H").mean().dropna()
hourly.head()

## Creación de características de calendario

In [ ]:
features = hourly.copy()
features["hour"] = features.index.hour
features["day_of_week"] = features.index.dayofweek
features["is_weekend"] = (features["day_of_week"] >= 5).astype(int)
features.head()

## Creación de variables rezagadas y medias móviles

In [ ]:
features["lag_1"] = features[gap_col].shift(1)
features["lag_24"] = features[gap_col].shift(24)
features["rolling_mean_24"] = features[gap_col].rolling(window=24, min_periods=1).mean()

features.dropna(inplace=True)
features.head()

## Visualización de patrones diarios promedio

In [ ]:
mean_by_hour = features.groupby("hour")[gap_col].mean()

plt.figure(figsize=(8, 4))
mean_by_hour.plot(marker="o")
plt.xlabel("Hora del día")
plt.ylabel(gap_col)
plt.title("Potencia activa promedio por hora del día")
plt.tight_layout()

## Guardado de dataset enriquecido para modelado

In [ ]:
output_path = Path("..") / "data" / "household_power_consumption_features_hourly.csv"
features.to_csv(output_path)
output_path